In [0]:
import requests
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

CEMAC_COUNTRIES = ["CMR", "CAF", "TCD", "COG", "GNQ", "GAB"]
INDICATOR_CODE = "NY.GDP.MKTP.CD"
DATE_RANGE = "1990:2024"

print(f"Catalog: cemac_ecowas_aes_trade")
print(f"Schema: bronze")
print(f"Target countries: {CEMAC_COUNTRIES}")
print(f"Indicator: {INDICATOR_CODE} (GDP, current US$)")
print(f"Date range: {DATE_RANGE}")

Catalog: cemac_ecowas_aes_trade
Schema: bronze
Target countries: ['CMR', 'CAF', 'TCD', 'COG', 'GNQ', 'GAB']
Indicator: NY.GDP.MKTP.CD (GDP, current US$)
Date range: 1990:2024


In [0]:
country_list = ";".join(CEMAC_COUNTRIES)
url = f"https://api.worldbank.org/v2/country/{country_list}/indicator/{INDICATOR_CODE}"
params = {"format": "json", "date": DATE_RANGE, "per_page": 1000}

print(f"Requesting: {url}")
print(f"Params: {params}")
print()

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()
metadata = data[0]
observations = data[1] if len(data) > 1 else []

print(f"HTTP status: {response.status_code}")
print(f"Pages: {metadata.get('pages')}, total observations: {metadata.get('total')}")
print(f"Observations actually returned: {len(observations)}")
print()
print("First observation (raw):")
print(observations[0] if observations else "NONE")

Requesting: https://api.worldbank.org/v2/country/CMR;CAF;TCD;COG;GNQ;GAB/indicator/NY.GDP.MKTP.CD
Params: {'format': 'json', 'date': '1990:2024', 'per_page': 1000}

HTTP status: 200
Pages: 1, total observations: 210
Observations actually returned: 210

First observation (raw):
{'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'}, 'country': {'id': 'CF', 'value': 'Central African Republic'}, 'countryiso3code': 'CAF', 'date': '2024', 'value': 2751494281.06887, 'unit': '', 'obs_status': '', 'decimal': 0}


In [0]:
# Record the current UTC time for data loading timestamp
loaded_at = datetime.utcnow()
rows = []

# Transform each observation into a Row object for Spark DataFrame
for obs in observations:
    rows.append(Row(
        country_iso3=obs["countryiso3code"],  # ISO3 country code
        country_name=obs["country"]["value"],  # Country name
        indicator_code=obs["indicator"]["id"],  # Indicator code
        indicator_name=obs["indicator"]["value"],  # Indicator name
        year=int(obs["date"]),  # Year of observation
        value=obs["value"],  # GDP value
        unit=obs.get("unit", ""),  # Unit (if available)
        obs_status=obs.get("obs_status", ""),  # Observation status (if available)
        source="worldbank_data360",  # Data source
        loaded_at=loaded_at,  # Timestamp when loaded
    ))

# Create Spark DataFrame from the list of Row objects
df = spark.createDataFrame(rows)

# Print DataFrame row count
print(f"DataFrame created with {df.count()} rows")
print()
# Print DataFrame schema
df.printSchema()
print()
# Show first 5 rows of the DataFrame
df.show(210)

/home/spark-fc4e83d6-e04f-4f00-a09a-f9/.ipykernel/2151/command-8657190760794586-267809363:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  loaded_at = datetime.utcnow()


DataFrame created with 210 rows

root
 |-- country_iso3: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- indicator_code: string (nullable = true)
 |-- indicator_name: string (nullable = true)
 |-- year: long (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- obs_status: string (nullable = true)
 |-- source: string (nullable = true)
 |-- loaded_at: timestamp (nullable = true)


+------------+--------------------+--------------+-----------------+----+-------------------+----+----------+-----------------+--------------------+
|country_iso3|        country_name|indicator_code|   indicator_name|year|              value|unit|obs_status|           source|           loaded_at|
+------------+--------------------+--------------+-----------------+----+-------------------+----+----------+-----------------+--------------------+
|         CAF|Central African R...|NY.GDP.MKTP.CD|GDP (current US$)|2024| 2.75149428106887E9|    |    

In [0]:
df.write.mode("overwrite").saveAsTable("bronze.data360_raw")

print("Write complete.")
print()

result = spark.sql("""
    SELECT COUNT(*) AS row_count,
           COUNT(value) AS non_null_values,
           MIN(year) AS earliest_year,
           MAX(year) AS latest_year,
           COUNT(DISTINCT country_iso3) AS countries_present
    FROM bronze.data360_raw
""")

result.show()

Write complete.

+---------+---------------+-------------+-----------+-----------------+
|row_count|non_null_values|earliest_year|latest_year|countries_present|
+---------+---------------+-------------+-----------+-----------------+
|      210|            210|         1990|       2024|                6|
+---------+---------------+-------------+-----------+-----------------+



In [0]:
# Analyze and display GDP data for CEMAC countries, including 2020 GDP, Cameroon's GDP trend, and Delta table history

print("CEMAC GDP in 2020 (latest pre-pandemic year with broad coverage):")
print()

spark.sql("""
    SELECT country_name,
           year,
           ROUND(value / 1e9, 2) AS gdp_usd_billions
    FROM bronze.data360_raw
    WHERE year = 2020
      AND value IS NOT NULL
    ORDER BY value DESC
""").show()

print()
print("Cameroon GDP trajectory (every 5 years):")
print()

spark.sql("""
    SELECT year,
           ROUND(value / 1e9, 2) AS gdp_usd_billions
    FROM bronze.data360_raw
    WHERE country_iso3 = 'CMR'
      AND year IN (1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023, 2024)
    ORDER BY year
""").show()

print()
print("Delta table history (proves time travel works):")
print()

spark.sql("DESCRIBE HISTORY bronze.data360_raw").select("version", "timestamp", "operation").show(truncate=False)

CEMAC GDP in 2020 (latest pre-pandemic year with broad coverage):

+--------------------+----+----------------+
|        country_name|year|gdp_usd_billions|
+--------------------+----+----------------+
|            Cameroon|2020|           40.77|
|               Gabon|2020|           15.34|
|                Chad|2020|           14.93|
|         Congo, Rep.|2020|           11.47|
|   Equatorial Guinea|2020|            9.89|
|Central African R...|2020|            2.33|
+--------------------+----+----------------+


Cameroon GDP trajectory (every 5 years):

+----+----------------+
|year|gdp_usd_billions|
+----+----------------+
|1990|           12.31|
|1995|           10.86|
|2000|           10.57|
|2005|           19.51|
|2010|           27.51|
|2015|           32.21|
|2020|           40.77|
|2023|           48.81|
|2024|            53.3|
+----+----------------+


Delta table history (proves time travel works):

+-------+-------------------+---------------------------------+
|version|tim

In [0]:
print("2024 coverage check:")
spark.sql("""
    SELECT country_name,
           year,
           CASE WHEN value IS NULL THEN 'missing'
                ELSE CONCAT('$', ROUND(value/1e9, 2), 'B')
           END AS gdp_2024
    FROM bronze.data360_raw
    WHERE year = 2024
    ORDER BY country_name
""").show(truncate=False)

2024 coverage check:
+------------------------+----+--------+
|country_name            |year|gdp_2024|
+------------------------+----+--------+
|Cameroon                |2024|$53.3B  |
|Central African Republic|2024|$2.75B  |
|Chad                    |2024|$19.52B |
|Congo, Rep.             |2024|$15.72B |
|Equatorial Guinea       |2024|$12.77B |
|Gabon                   |2024|$20.9B  |
+------------------------+----+--------+

